<a href="https://colab.research.google.com/github/DLuisMontufar3/1/blob/main/Si7_iipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*07 SEGMENTACION DE DATOS MODELO RFM*


Paso 1: Generacion carga y Exploracion de Datos

In [ ]:
import pandas as pd
import numpy as np
import datetime

Transformar el registro histórico de transacciones (datos crudos) en una Matriz de Segmentación RFM estratégica para responder:

¿Quiénes son nuestros clientes "Campeones"? (Compran mucho y seguido).
¿Quiénes son nuestros clientes "En Riesgo"? (Compraban mucho antes, pero se han ido).
¿Quiénes son "Leales"? (Compran seguido).

In [ ]:
np.random.seed(42)
n_transacciones = 1000
n_clientes = 50

In [ ]:
fechas = pd.date_range(start='2023-01-01', end='2023-12-31', periods=n_transacciones)
fechas = np.random.choice(fechas, n_transacciones) # Desordenar fechas
cliente_ids = np.random.randint(1000, 1000 + n_clientes, n_transacciones)
cantidades = np.random.randint(1, 10, n_transacciones)
precios = np.round(np.random.uniform(10, 100, n_transacciones), 2)

In [ ]:
# Crear DataFrame
df = pd.DataFrame({
    'ID_Transaccion': [f'TRX-{i}' for i in range(n_transacciones)],
    'ID_Cliente': cliente_ids,
    'Fecha': fechas,
    'Cantidad': cantidades,
    'Precio_Unitario': precios
})

# Calcular Total y Guardar
df['Total'] = df['Cantidad'] * df['Precio_Unitario']

In [ ]:
df

,ID_Transaccion,ID_Cliente,Fecha,Cantidad,Precio_Unitario,Total
0,TRX-0,1044,2023-02-07 03:57:50.270270270,5,30.28,151.40
1,TRX-1,1004,2023-06-08 11:57:50.270270270,9,32.59,293.31
2,TRX-2,1032,2023-11-10 08:28:49.729729732,3,86.56,259.68
3,TRX-3,1000,2023-04-09 09:04:51.891891892,7,60.51,423.57
4,TRX-4,1017,2023-02-08 14:56:34.594594594,5,57.11,285.55
...,...,...,...,...,...,...
995,TRX-995,1041,2023-01-04 06:42:09.729729729,9,19.12,172.08
996,TRX-996,1035,2023-10-27 20:55:29.729729732,9,35.01,315.09
997,TRX-997,1026,2023-10-18 09:33:41.621621624,1,34.87,34.87
998,TRX-998,1037,2023-03-29 19:29:00.540540541,2,48.88,97.76


2.1 Recencia

In [ ]:
df['Fecha'].max()

Timestamp('2023-12-30 15:15:18.918918920')

In [ ]:
df.groupby('ID_Cliente')['Fecha'].max()

,Fecha
ID_Cliente,
1000,2023-12-20 01:39:27.567567568
1001,2023-12-29 21:45:56.756756756
1002,2023-12-29 21:45:56.756756756
1003,2023-12-22 06:07:34.054054056
1004,2023-12-24 10:35:40.540540540
1005,2023-12-29 13:01:15.675675676
1006,2023-11-30 00:41:48.108108108
1007,2023-12-30 15:15:18.918918920
1008,2023-12-21 12:38:11.891891892


In [ ]:
df['Fecha'] = pd.to_datetime(df['Fecha'])

In [ ]:
df_rfm = df.groupby('ID_Cliente')['Fecha'].max()


In [ ]:
#df_rfm = pd.DataFrame(groupby('ID_Cliente')['Fecha'].max())

In [ ]:
hoy = pd.to_datetime('today')

rfm = df.groupby('ID_Cliente')['Fecha'].max().rsub(hoy).dt.days.to_frame('Recencia')

rfm

,Recencia
ID_Cliente,
1000,757
1001,748
1002,748
1003,755
1004,753
1005,748
1006,777
1007,747
1008,756


In [ ]:

frecuencia = df.groupby('ID_Cliente')['ID_Transaccion'].nunique().to_frame('Frecuencia')

frecuencia.head()

,Frecuencia
ID_Cliente,
1000,21
1001,18
1002,21
1003,20
1004,25


In [ ]:
monetario = df.groupby('ID_Cliente')['Total'].sum().to_frame('Monetario')

monetario.head()

,Monetario
ID_Cliente,
1000,5304.32
1001,5528.83
1002,4389.77
1003,4860.34
1004,8050.55


In [ ]:
rfm = rfm.join(frecuencia).join(monetario)
rfm.head()

,Recencia,Frecuencia,Monetario
ID_Cliente,,,
1000,757,21,5304.32
1001,748,18,5528.83
1002,748,21,4389.77
1003,755,20,4860.34
1004,753,25,8050.55


In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recencia'], 3, labels=[3,2,1])
rfm[['Recencia','R_Score']].head()

,Recencia,R_Score
ID_Cliente,,
1000,757,2
1001,748,3
1002,748,3
1003,755,2
1004,753,3


In [ ]:
rfm['F_Score'] = pd.qcut(rfm['Frecuencia'], 3, labels=[1,2,3])
rfm[['Frecuencia','F_Score']].head()

,Frecuencia,F_Score
ID_Cliente,,
1000,21,2
1001,18,1
1002,21,2
1003,20,2
1004,25,3


In [ ]:
rfm['M_Score'] = pd.qcut(rfm['Monetario'], 3, labels=[1,2,3])
rfm[['Monetario','M_Score']].head()

,Monetario,M_Score
ID_Cliente,,
1000,5304.32,2
1001,5528.83,2
1002,4389.77,1
1003,4860.34,1
1004,8050.55,3


In [ ]:
print(rfm.head())

            Recencia  Frecuencia  Monetario R_Score F_Score M_Score
ID_Cliente                                                         
1000             757          21    5304.32       2       2       2
1001             748          18    5528.83       3       1       2
1002             748          21    4389.77       3       2       1
1003             755          20    4860.34       2       2       1
1004             753          25    8050.55       3       3       3


In [ ]:
segmentos = [ 'Campeones', 'Leales', 'En Riesgo', 'Perdidos', 'Nuevos' ]

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recencia'], 3, labels=[3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frecuencia'], 3, labels=[1,2,3]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetario'], 3, labels=[1,2,3]).astype(int)

In [ ]:
condiciones = [ (rfm['R_Score'] == 3) & (rfm['F_Score'] == 3) & (rfm['M_Score'] == 3), # campeones
                 (rfm['F_Score'] == 3), # leales
                 (rfm['R_Score'] == 1) & ((rfm['F_Score'] == 3) | (rfm['M_Score'] == 3)), # en Riesgo
                 (rfm['R_Score'] == 1) & (rfm['F_Score'] == 1) & (rfm['M_Score'] == 1), # perdidos
                 (rfm['R_Score'] == 3) & (rfm['F_Score'] == 1) # nuevos
                 ]

In [ ]:
rfm['Segmento'] = np.select(condiciones, segmentos, default='Otros')

In [ ]:
print(rfm[['Recencia','R_Score','Frecuencia','F_Score','Monetario','M_Score','Segmento']].head())

            Recencia  R_Score  Frecuencia  F_Score  Monetario  M_Score  \
ID_Cliente                                                               
1000             757        2          21        2    5304.32        2   
1001             748        3          18        1    5528.83        2   
1002             748        3          21        2    4389.77        1   
1003             755        2          20        2    4860.34        1   
1004             753        3          25        3    8050.55        3   

             Segmento  
ID_Cliente             
1000            Otros  
1001           Nuevos  
1002            Otros  
1003            Otros  
1004        Campeones  


In [ ]:
rfm


,Recencia,Frecuencia,Monetario,R_Score,F_Score,M_Score,Segmento
ID_Cliente,,,,,,,
1000,757,21,5304.32,2,2,2,Otros
1001,748,18,5528.83,3,1,2,Nuevos
1002,748,21,4389.77,3,2,1,Otros
1003,755,20,4860.34,2,2,1,Otros
1004,753,25,8050.55,3,3,3,Campeones
1005,748,22,6076.92,3,3,3,Campeones
1006,777,15,4359.10,1,1,1,Perdidos
1007,747,20,6304.75,3,2,3,Otros
1008,756,21,4970.26,2,2,2,Otros


In [ ]:
print(rfm['Segmento'].value_counts())



Segmento
Otros        23
Leales        8
Campeones     7
Perdidos      7
Nuevos        5
Name: count, dtype: int64
